In [1]:
import cv2
import numpy as np
from ultralytics import YOLO
import torch
import os
from collections import deque
from catboost import CatBoostRegressor

In [ ]:
cnt = 0
cap = cv2.VideoCapture("VID_20260412_174201.mp4")
while cap.isOpened():
    ret , frame = cap.read()
    if not ret:
        break
    if cnt % 10 == 0: # каждый 10 кадр
        cv2.imwrite(f"images/{cnt}.jpg", frame)
    cnt += 1
cap.release()

In [11]:
input_dir = 'images' # все по hsv
output_dir = 'labels'
LOWER_BLUE = np.array([85, 100, 100]) 
UPPER_BLUE = np.array([125, 255, 255])
for i in os.listdir(input_dir):
    img = cv2.imread(os.path.join(input_dir, i)) # склейка пути
    h, w, _ = img.shape # _ это заглушка 
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV) 
    mask = cv2.inRange(hsv, LOWER_BLUE, UPPER_BLUE) 
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE) # ищет белый цвет
    coords = [] # для хранения координат как требует yolo
    for cnt in contours:    
        bx, by, bw, bh = cv2.boundingRect(cnt) # описывает контур в квадрат
        if cv2.contourArea(cnt) < 20: # убираем мелкий шум
            continue
        coords.append(f"0 {(bx+bw/2)/w:.6f} {(by+bh/2)/h} {bw/w} {bh/h}") # так требует yolo
    with open(os.path.join(output_dir, os.path.splitext(i)[0] + ".txt"), "w") as f: # генерит имя файла и делает доступным для записи
        f.write("\n".join(coords)) # сохраняет найденные координаты

In [ ]:
model = YOLO('yolo26s.pt')
model.train(
    data = r'C:\Users\user\projectolymp\data.yaml',
    epochs=100,
    imgsz=640, 
    batch=20,
    device="cuda" if torch.cuda.is_available() else "cpu", 
    project="models",
    hsv_h=0.015, # аугментации
    hsv_s=0.7,
    hsv_v=0.4,
    mosaic=1.0,
    degrees=15.0,
    translate=0.1, 
    fliplr=0.54,
    close_mosaic=10 
)

In [ ]:
model = YOLO(r'C:\Users\user\projectolymp\runs\detect\models\train\weights\best.pt') 
cb_model = CatBoostRegressor()
cb_model.load_model('catboost.cbm')
cap = cv2.VideoCapture(0)
track_history = {}

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: # на всякий случай
        break
    results = model.track(frame, persist=True, conf=0.6) # ищет маркеры и присваевает им id
    if results[0].boxes.id is not None: # проверка нашла ли модель хоть 1 маркер
        boxes = results[0].boxes.xywh.cpu().numpy() # получение координатов рамок объектов
        track_ids = results[0].boxes.id.int().cpu().tolist() # получение id всегда одинаковый т.к. у нас детектится только 1 метка
        for box, track_id in zip(boxes, track_ids): # проходим по координате и ее id
            x, y, w, h = box # распаковка
            if track_id not in track_history: 
                track_history[track_id] = deque(maxlen=150)
            track = track_history[track_id]
            track.append((float(x), float(y)))
            for i in range(1, len(track)):
                pt1 = (int(track[i-1][0]), int(track[i-1][1]))
                pt2 = (int(track[i][0]), int(track[i][1]))
                cv2.line(frame, pt1, pt2, (0, 0, 255), 2)
            if len(track) >= 10:
                current_past = list(track)[-10:]
                input_deltas = []
                for i in range(1, len(current_past)):
                    input_deltas.append(current_past[i][0] - current_past[i-1][0]) 
                    input_deltas.append(current_past[i][1] - current_past[i-1][1]) 
                prediction = cb_model.predict([input_deltas])
                pred_dx, pred_dy = prediction[0]
                target_x = int(current_past[-1][0] + pred_dx)
                target_y = int(current_past[-1][1] + pred_dy)
                target_x = np.clip(target_x, 0, frame.shape[1])
                target_y = np.clip(target_y, 0, frame.shape[0])
                cv2.circle(frame, (target_x, target_y), 7, (255, 0, 0), -1)
                cv2.line(frame, (int(current_past[-1][0]), int(current_past[-1][1])), 
                         (target_x, target_y), (255, 255, 0), 1)

    cv2.imshow("MMC", frame)
    if cv2.waitKey(1) & 0xFF == ord('1'):
        break

cap.release()
cv2.destroyAllWindows()